# 📊 01 — Data Exploration

**EvaliSense — AI-Powered Handwritten Examination Evaluation**

This notebook explores all available datasets used in the project:
1. **IAM Handwriting Lines** — 10,000+ real handwritten text images with transcriptions
2. **Sample Rubrics** — Structured marking schemes across 4 subjects
3. **Sample Answers** — Typed student answers at varying quality levels
4. **Synthetic Training Data** — 300 records for ML model development
5. **Test Images** — Programmatic handwriting-style images

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

DATA_DIR = Path('../data')
print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Exists: {DATA_DIR.exists()}')

## 1. Dataset Overview

Let's first map out everything we have available.

In [ ]:
def count_files(directory, extensions=('.png', '.jpg', '.jpeg')):
    """Count image files recursively."""
    if not directory.exists():
        return 0
    return sum(1 for f in directory.rglob('*') if f.suffix.lower() in extensions)

datasets = {
    'IAM Lines (train)': DATA_DIR / 'external' / 'iam_lines' / 'train',
    'IAM Lines (test)': DATA_DIR / 'external' / 'iam_lines' / 'test',
    'IAM Lines (validation)': DATA_DIR / 'external' / 'iam_lines' / 'validation',
    'Raw Test Images': DATA_DIR / 'raw',
}

print('=== Image Dataset Summary ===')
print(f'{"Dataset":<30} {"Images":>8}  {"Path"}')
print('-' * 80)
total = 0
for name, path in datasets.items():
    count = count_files(path)
    total += count
    status = '✓' if count > 0 else '✗'
    print(f'{status} {name:<28} {count:>8}  {path}')
print(f'\nTotal images: {total}')

# Non-image data
print('\n=== Other Data ===')
rubrics = list((DATA_DIR / 'samples').glob('*.json'))
print(f'Sample rubrics: {len(rubrics)}')
for r in rubrics:
    print(f'  • {r.name}')

synthetic = DATA_DIR / 'samples' / 'synthetic_dataset.jsonl'
if synthetic.exists():
    with open(synthetic) as f:
        n_synthetic = sum(1 for _ in f)
    print(f'Synthetic training records: {n_synthetic}')

model_path = Path('../trained_models/risk_model.joblib')
print(f'Trained model: {"✓" if model_path.exists() else "✗"}')

## 2. IAM Handwriting Dataset Exploration

The IAM dataset contains real handwritten English text lines — the standard benchmark for Handwritten Text Recognition (HTR).

In [ ]:
# Load IAM metadata
iam_meta_path = DATA_DIR / 'external' / 'iam_lines' / 'metadata.jsonl'
iam_records = []

if iam_meta_path.exists():
    with open(iam_meta_path, 'r', encoding='utf-8') as f:
        for line in f:
            iam_records.append(json.loads(line.strip()))
    
    print(f'Total IAM records: {len(iam_records)}')
    
    # Split distribution
    splits = Counter(r['split'] for r in iam_records)
    print(f'\nSplit distribution:')
    for split, count in splits.most_common():
        print(f'  {split}: {count}')
    
    # Transcription length stats
    lengths = [len(r.get('transcription', '')) for r in iam_records]
    print(f'\nTranscription length stats:')
    print(f'  Min: {min(lengths)} chars')
    print(f'  Max: {max(lengths)} chars')
    print(f'  Mean: {np.mean(lengths):.1f} chars')
    print(f'  Median: {np.median(lengths):.1f} chars')
else:
    print('IAM metadata not found. Run: python download_exam_datasets.py --dataset iam')

In [ ]:
# Visualize sample IAM images
if iam_records:
    fig, axes = plt.subplots(4, 2, figsize=(16, 10))
    fig.suptitle('Sample IAM Handwriting Line Images', fontsize=16, fontweight='bold')
    
    samples = iam_records[:8]
    for idx, (ax, rec) in enumerate(zip(axes.flat, samples)):
        img_path = DATA_DIR / rec['image_file']
        if img_path.exists():
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            ax.imshow(img, cmap='gray')
            ax.set_title(f'GT: "{rec["transcription"][:50]}"', fontsize=9)
        else:
            ax.text(0.5, 0.5, 'Image not found', ha='center', va='center')
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('../data/results/iam_samples_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved visualization to data/results/iam_samples_visualization.png')

In [ ]:
# Transcription length distribution
if iam_records:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Character length distribution
    char_lengths = [len(r.get('transcription', '')) for r in iam_records]
    ax1.hist(char_lengths, bins=40, color='#4A90D9', edgecolor='white', alpha=0.8)
    ax1.set_xlabel('Transcription Length (characters)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Transcription Lengths')
    ax1.axvline(np.mean(char_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(char_lengths):.0f}')
    ax1.legend()
    
    # Word count distribution
    word_counts = [len(r.get('transcription', '').split()) for r in iam_records]
    ax2.hist(word_counts, bins=30, color='#E8744F', edgecolor='white', alpha=0.8)
    ax2.set_xlabel('Word Count')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Distribution of Word Counts per Line')
    ax2.axvline(np.mean(word_counts), color='red', linestyle='--', label=f'Mean: {np.mean(word_counts):.1f}')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig('../data/results/iam_length_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Sample Rubrics Exploration

Rubrics define the marking criteria for each question. Let's examine their structure.

In [ ]:
rubric_dir = DATA_DIR / 'samples'
rubric_files = sorted(rubric_dir.glob('*.json'))
# Exclude sample_answers.json
rubric_files = [f for f in rubric_files if 'answer' not in f.name]

rubrics_data = []
for rubric_file in rubric_files:
    with open(rubric_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    rubrics_data.append(data)
    
    print(f'\n📝 {rubric_file.name}')
    print(f'   Subject: {data.get("metadata", {}).get("subject", "Unknown")}')
    print(f'   Topic: {data.get("metadata", {}).get("topic", "Unknown")}')
    print(f'   Max Marks: {data["max_marks"]}')
    print(f'   Criteria: {len(data["criteria"])}')
    print(f'   Question: {data["question"][:80]}...')
    for c in data['criteria']:
        keywords = ', '.join(c.get('keywords', [])[:4])
        print(f'     [{c["id"]}] {c["description"][:50]} ({c["marks"]} marks) — keywords: {keywords}')

In [ ]:
# Rubric comparison chart
if rubrics_data:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    subjects = [r.get('metadata', {}).get('subject', 'Unknown') for r in rubrics_data]
    max_marks = [r['max_marks'] for r in rubrics_data]
    n_criteria = [len(r['criteria']) for r in rubrics_data]
    
    colors = ['#4A90D9', '#E8744F', '#50C878', '#9B59B6']
    
    ax1.bar(subjects, max_marks, color=colors[:len(subjects)], edgecolor='white')
    ax1.set_ylabel('Max Marks')
    ax1.set_title('Maximum Marks by Subject')
    for i, v in enumerate(max_marks):
        ax1.text(i, v + 0.3, str(v), ha='center', fontweight='bold')
    
    ax2.bar(subjects, n_criteria, color=colors[:len(subjects)], edgecolor='white')
    ax2.set_ylabel('Number of Criteria')
    ax2.set_title('Marking Criteria Count by Subject')
    for i, v in enumerate(n_criteria):
        ax2.text(i, v + 0.1, str(v), ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../data/results/rubric_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Sample Answers Exploration

In [ ]:
answers_path = DATA_DIR / 'samples' / 'sample_answers.json'
if answers_path.exists():
    with open(answers_path, 'r', encoding='utf-8') as f:
        answers_data = json.load(f)
    
    answers = answers_data['answers']
    print(f'Total sample answers: {len(answers)}')
    print(f'\n{"ID":<20} {"Quality":<10} {"Rubric":<35} {"Score Range":<12} {"Length":>6}')
    print('-' * 90)
    for a in answers:
        score_range = f"{a['expected_score_range'][0]}-{a['expected_score_range'][1]}"
        length = len(a['answer_text'])
        print(f"{a['id']:<20} {a['quality']:<10} {a['rubric_file']:<35} {score_range:<12} {length:>6}")
else:
    print('Sample answers not found.')

## 5. Synthetic Training Data Analysis

In [ ]:
synthetic_path = DATA_DIR / 'samples' / 'synthetic_dataset.jsonl'
if synthetic_path.exists():
    records = []
    with open(synthetic_path, 'r', encoding='utf-8') as f:
        for line in f:
            records.append(json.loads(line.strip()))
    
    print(f'Synthetic records: {len(records)}')
    
    errors = [r for r in records if r['is_grading_error']]
    non_errors = [r for r in records if not r['is_grading_error']]
    print(f'Grading errors: {len(errors)} ({len(errors)/len(records):.1%})')
    print(f'Non-errors: {len(non_errors)} ({len(non_errors)/len(records):.1%})')
    
    # Feature distributions
    ai_marks = [r['ai_mark'] for r in records]
    expert_marks = [r['expert_mark'] for r in records]
    abs_errors = [r['absolute_error'] for r in records]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # AI vs Expert marks
    ax = axes[0, 0]
    colors_scatter = ['#E8744F' if r['is_grading_error'] else '#4A90D9' for r in records]
    ax.scatter(ai_marks, expert_marks, c=colors_scatter, alpha=0.5, s=20)
    ax.plot([0, 20], [0, 20], 'k--', alpha=0.3, label='Perfect agreement')
    ax.set_xlabel('AI Mark')
    ax.set_ylabel('Expert Mark')
    ax.set_title('AI Mark vs Expert Mark')
    ax.legend()
    
    # Absolute error distribution
    ax = axes[0, 1]
    ax.hist(abs_errors, bins=30, color='#4A90D9', edgecolor='white', alpha=0.8)
    ax.axvline(2.0, color='red', linestyle='--', linewidth=2, label='Error threshold (2.0)')
    ax.set_xlabel('Absolute Error (|AI - Expert|)')
    ax.set_ylabel('Frequency')
    ax.set_title('Distribution of Grading Errors')
    ax.legend()
    
    # Class balance
    ax = axes[1, 0]
    labels = ['Non-Error', 'Grading Error']
    sizes = [len(non_errors), len(errors)]
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=['#4A90D9', '#E8744F'],
           startangle=90, explode=(0, 0.05))
    ax.set_title('Class Distribution')
    
    # Feature correlation with error
    ax = axes[1, 1]
    feature_keys = ['ocr_confidence', 'overall_semantic_similarity', 'evaluation_confidence',
                    'rubric_coverage', 'normalized_score']
    error_means = [np.mean([r['features'][k] for r in errors]) for k in feature_keys]
    non_error_means = [np.mean([r['features'][k] for r in non_errors]) for k in feature_keys]
    
    x = np.arange(len(feature_keys))
    width = 0.35
    ax.bar(x - width/2, non_error_means, width, label='Non-Error', color='#4A90D9')
    ax.bar(x + width/2, error_means, width, label='Grading Error', color='#E8744F')
    ax.set_xticks(x)
    ax.set_xticklabels([k.replace('_', '\n') for k in feature_keys], fontsize=8)
    ax.set_ylabel('Mean Value')
    ax.set_title('Feature Means: Error vs Non-Error')
    ax.legend()
    
    plt.suptitle('Synthetic Training Data Analysis', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../data/results/synthetic_data_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Synthetic dataset not found. Run: python generate_synthetic_dataset.py --count 300')

## 6. Test Images

Programmatically generated answer images for pipeline testing.

In [ ]:
raw_dir = DATA_DIR / 'raw'
test_images = sorted(raw_dir.glob('answer_*.png'))

if test_images:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Generated Test Answer Images', fontsize=16, fontweight='bold')
    
    for ax, img_path in zip(axes.flat, test_images):
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Show a cropped region (top portion)
        h = min(img_rgb.shape[0], 1500)
        ax.imshow(img_rgb[:h, :, :])
        ax.set_title(img_path.stem.replace('answer_', '').replace('_', ' ').title(), fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('../data/results/test_images_overview.png', dpi=100, bbox_inches='tight')
    plt.show()
else:
    print('No test images found. Run: python create_test_images.py')

## Summary

| Dataset | Records | Type | Purpose |
|---------|---------|------|---------|
| IAM Handwriting Lines | 10,000+ | Real handwriting images + transcriptions | HTR benchmarking |
| Sample Rubrics | 4 | JSON schemas | Evaluation criteria (Biology, History, Physics) |
| Sample Answers | 8 | Typed text | Evaluation pipeline testing |
| Synthetic Dataset | 300 | Feature vectors + labels | ML model training |
| Test Images | 6 | OpenCV-rendered pages | End-to-end pipeline testing |

**Next:** Proceed to `02_preprocessing_pipeline.ipynb` to explore image preprocessing.